In [0]:
# Databricks notebook source

# COMMAND ----------
# Configuración

CATALOG_NAME  = "workspace"
SCHEMA_NAME   = "default"
SOURCE_TABLE  = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_retail_sales"

DIM_CUSTOMER  = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_customer"
DIM_ITEM      = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_item"
DIM_DATE      = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_date"

DATE_START    = "2022-01-01"
DATE_END      = "2025-01-18"

# COMMAND ----------
# Lectura desde Silver

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table(SOURCE_TABLE)
print(f"Filas Silver: {silver.count():,}")

# COMMAND ----------
# dim_customer

w_customer = Window.orderBy("customer_id")

dim_customer = (
    silver
    .select("customer_id")
    .distinct()
    .withColumn("customer_key",  F.row_number().over(w_customer).cast("bigint"))
    .withColumn("customer_code", F.concat(F.lit("CUST-"), F.col("customer_key").cast("string")))
    .select("customer_key", "customer_id", "customer_code")
)

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_CUSTOMER)
)

count_customer = spark.table(DIM_CUSTOMER).count()
dups_customer  = count_customer - spark.table(DIM_CUSTOMER).select("customer_id").distinct().count()
print(f"dim_customer — filas: {count_customer:,} | duplicados NK: {dups_customer}")
assert dups_customer == 0, "customer_id no es único en dim_customer"
print("✓ dim_customer OK")

# COMMAND ----------
# dim_item

w_item = Window.orderBy("item_name")

dim_item = (
    silver
    .select(
        F.col("item").alias("item_name"),
        F.col("category")
    )
    .distinct()
    .withColumn("item_key",      F.row_number().over(w_item).cast("bigint"))
    .withColumn("item_id",       F.concat(F.lit("ITEM-"), F.col("item_key").cast("string")))
    .withColumn("category_code", F.concat(F.lit("CAT-"),
                                  F.dense_rank().over(Window.orderBy("category")).cast("string")))
    .select("item_key", "item_id", "item_name", "category", "category_code")
)

(
    dim_item.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_ITEM)
)

count_item = spark.table(DIM_ITEM).count()
dups_item  = count_item - spark.table(DIM_ITEM).select("item_name").distinct().count()
print(f"dim_item — filas: {count_item:,} | duplicados NK: {dups_item}")
assert dups_item == 0, "item_name no es único en dim_item"
print("✓ dim_item OK")

# COMMAND ----------
# dim_date — generación programática

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, BooleanType
import datetime

MONTHS_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}
DAYS_ES = {
    0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves",
    4: "Viernes", 5: "Sábado", 6: "Domingo"
}

start = datetime.date.fromisoformat(DATE_START)
end   = datetime.date.fromisoformat(DATE_END)
rows  = []
current = start

while current <= end:
    dow = current.weekday()          # 0=Lunes … 6=Domingo
    rows.append((
        int(current.strftime("%Y%m%d")),   # date_key
        current,                            # full_date
        current.year,                       # year
        (current.month - 1) // 3 + 1,      # quarter
        current.month,                      # month
        MONTHS_ES[current.month],           # month_name
        current.day,                        # day
        dow + 1,                            # day_of_week (1=Lunes … 7=Domingo)
        DAYS_ES[dow],                       # day_name
        dow >= 5                            # is_weekend
    ))
    current += datetime.timedelta(days=1)

schema_date = StructType([
    StructField("date_key",    IntegerType(), False),
    StructField("full_date",   DateType(),    False),
    StructField("year",        IntegerType(), False),
    StructField("quarter",     IntegerType(), False),
    StructField("month",       IntegerType(), False),
    StructField("month_name",  StringType(),  False),
    StructField("day",         IntegerType(), False),
    StructField("day_of_week", IntegerType(), False),
    StructField("day_name",    StringType(),  False),
    StructField("is_weekend",  BooleanType(), False),
])

dim_date = spark.createDataFrame(rows, schema=schema_date)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_DATE)
)

count_date = spark.table(DIM_DATE).count()
dups_date  = count_date - spark.table(DIM_DATE).select("date_key").distinct().count()
print(f"dim_date — filas: {count_date:,} | duplicados PK: {dups_date}")
assert dups_date == 0, "date_key no es único en dim_date"
print("✓ dim_date OK")

# COMMAND ----------
# Validación cruzada — fechas de Silver cubiertas por dim_date

fechas_silver   = silver.select(F.col("transaction_date").cast("date").alias("full_date")).distinct()
fechas_dim_date = spark.table(DIM_DATE).select("full_date")

sin_cobertura = fechas_silver.join(fechas_dim_date, on="full_date", how="left_anti").count()
print(f"Fechas en Silver sin cobertura en dim_date: {sin_cobertura}")
assert sin_cobertura == 0, "Hay fechas en Silver fuera del rango de dim_date"
print("✓ Cobertura de fechas OK")

# COMMAND ----------
# Muestras finales

print("── dim_customer (5 filas) ──")
display(spark.table(DIM_CUSTOMER).limit(5))

print("── dim_item (5 filas) ──")
display(spark.table(DIM_ITEM).limit(5))

print("── dim_date (5 filas) ──")
display(spark.table(DIM_DATE).orderBy("date_key").limit(5))

Filas Silver: 11,362


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_customer — filas: 25 | duplicados NK: 0
✓ dim_customer OK


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_item — filas: 200 | duplicados NK: 0
✓ dim_item OK
dim_date — filas: 1,114 | duplicados PK: 0
✓ dim_date OK
Fechas en Silver sin cobertura en dim_date: 0
✓ Cobertura de fechas OK
── dim_customer (5 filas) ──


...

── dim_item (5 filas) ──


item_key,item_id,item_name,category,category_code
1,ITEM-1,Item_10_BEV,Beverages,CAT-1
9,ITEM-9,Item_11_BEV,Beverages,CAT-1
17,ITEM-17,Item_12_BEV,Beverages,CAT-1
25,ITEM-25,Item_13_BEV,Beverages,CAT-1
33,ITEM-33,Item_14_BEV,Beverages,CAT-1


── dim_date (5 filas) ──


date_key,full_date,year,quarter,month,month_name,day,day_of_week,day_name,is_weekend
20220101,2022-01-01,2022,1,1,Enero,1,6,Sábado,true
20220102,2022-01-02,2022,1,1,Enero,2,7,Domingo,true
20220103,2022-01-03,2022,1,1,Enero,3,1,Lunes,false
20220104,2022-01-04,2022,1,1,Enero,4,2,Martes,false
20220105,2022-01-05,2022,1,1,Enero,5,3,Miércoles,false


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
